# 기존 Qdrant 청크 → C Hybrid → D Reranking → 근거 답변
터미널에서 설치 후 커널을 재시작하세요:
`uv add rank-bm25 sentence-transformers "transformers>=4.51.0"`
이미 저장된 청크를 그대로 사용합니다. 질문 임베딩만 호출합니다.
Qwen 첫 실행은 모델 다운로드·로딩이 필요하며 CPU에서는 느릴 수 있습니다.


In [ ]:
from pathlib import Path
import sys, re
from collections import Counter
from dotenv import load_dotenv

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'common/ai_model.py').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('프로젝트 폴더에서 실행하세요.')
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / '.env', override=True)
from common.ai_model import get_embedding_model, get_llm_model
from common.qdrant import get_qdrant_client
from rank_bm25 import BM25Okapi

embedding_client = get_embedding_model()  # 저장 당시와 동일한 모델이어야 합니다.
qdrant_client = get_qdrant_client()
COLLECTION_NAME = 'ai_basic_law_article_paragraph_v1'
print('사용 가능한 컬렉션:', [c.name for c in qdrant_client.get_collections().collections])


사용 가능한 컬렉션: ['ai_basic_law_article_paragraph_v1', 'ai_basic_law_articles_v1', 'ai_basic_law_xml', 'law_articles']


## 1. 기존 컬렉션 읽기
다른 이름으로 저장했다면 위 COLLECTION_NAME만 변경하세요. 모든 페이지를 읽고 포인트 ID를 RRF 공통 ID로 사용합니다.


In [2]:
if not qdrant_client.collection_exists(COLLECTION_NAME):
    raise ValueError(f'컬렉션이 없습니다: {COLLECTION_NAME}. 저장 당시 이름을 지정하세요.')
chunks = []
offset = None
while True:
    records, offset = qdrant_client.scroll(
        collection_name=COLLECTION_NAME, limit=256, offset=offset,
        with_payload=True, with_vectors=False)
    for point in records:
        payload = dict(point.payload or {})
        if not isinstance(payload.get('content'), str) or not payload['content'].strip():
            continue
        payload.update(chunk_id=str(point.id))
        chunks.append(payload)
    if offset is None:
        break
if not chunks:
    raise ValueError('content가 있는 저장 문서가 없습니다.')
print('불러온 청크:', len(chunks))
print('유형별 개수:', dict(Counter(c.get('chunk_type', 'unknown') for c in chunks)))
print('제2조 청크 수:', sum(c.get('article') in ('제2조', '2') for c in chunks))


ValueError: 컬렉션이 없습니다: ai_basic_law_article_plus_paragraph_v1. 저장 당시 이름을 지정하세요.

In [ ]:
# C: Dense + BM25 + RRF. 한국어 조사 일부를 정규화합니다.
def tokenize(text):
    tokens = re.findall(r'[가-힣]+|[a-z]+|\d+', text.lower())
    return [re.sub(r'(이란|이랑|인가요|은|는|을|를|의)$', '', t) if len(t) > 3 else t for t in tokens]

bm25 = BM25Okapi([tokenize(c.get('title', '') + ' ' + c['content']) for c in chunks])
async def hybrid_search(query, candidate_k=20):
    vector = await embedding_client.aembed_query(query)
    dense = qdrant_client.query_points(collection_name=COLLECTION_NAME, query=vector, limit=candidate_k).points
    dense_docs = [{**p.payload, 'chunk_id': str(p.id)} for p in dense if p.payload and p.payload.get('content')]
    scores = bm25.get_scores(tokenize(query))
    keyword_docs = [chunks[i] for i in sorted(range(len(chunks)), key=lambda i: scores[i], reverse=True)
                    if scores[i] > 0][:candidate_k]
    totals, by_id = {}, {}
    for ranking in [dense_docs, keyword_docs]:
        for rank, c in enumerate(ranking, 1):
            cid = c['chunk_id']
            totals[cid] = totals.get(cid, 0) + 1 / (60 + rank)
            by_id[cid] = c
    hybrid = [{**by_id[cid], 'rrf_score': totals[cid]}
              for cid in sorted(totals, key=totals.get, reverse=True)[:candidate_k]]
    return dense_docs, hybrid


In [ ]:
# D: Qwen reranker. 최초 호출에만 모델을 로드하고 이후 재사용합니다.
import torch
from sentence_transformers import CrossEncoder

reranker = None
def rerank(query, candidates, top_k=5):
    global reranker
    if not candidates:
        return []
    if reranker is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print('Qwen 모델 로딩 / device:', device)
        reranker = CrossEncoder('Qwen/Qwen3-Reranker-0.6B', device=device, max_length=4096)
    pairs = [(query, f"{c.get('article', '')} {c.get('title', '')}\n{c['content']}") for c in candidates]
    scores = reranker.predict(pairs, batch_size=1, activation_fn=torch.nn.Sigmoid(), show_progress_bar=True)
    return sorted([{**c, 'rerank_score': float(s)} for c, s in zip(candidates, scores)],
                  key=lambda c: c['rerank_score'], reverse=True)[:top_k]


In [ ]:
# 질문마다 이 셀을 재실행합니다. USE_RERANK=False이면 C만 비교합니다.
import time
query = '고영향 인공지능이란 무엇인가요?'
USE_RERANK = True
started = time.perf_counter()
dense_results, hybrid_results = await hybrid_search(query)
final_results = rerank(query, hybrid_results) if USE_RERANK else hybrid_results[:5]
for label, rows in [('Dense', dense_results[:5]), ('C Hybrid', hybrid_results[:5]), ('C + D' if USE_RERANK else '최종', final_results)]:
    print('\n', label)
    for c in rows:
        print(c.get('chunk_type', ''), c.get('article', ''), c.get('paragraph', ''),
              c.get('title', ''), 'rerank=', c.get('rerank_score'), 'rrf=', c.get('rrf_score'))
        print(c['content'][:1000])
print('검색 시간(초):', round(time.perf_counter()-started, 2))


In [ ]:
# 저장된 조문 전체가 있으면 부모 Context로 사용하고, 없으면 검색 청크를 그대로 사용합니다.
article_parents = {
    (c.get('law_name', ''), c.get('effective_date', ''), c.get('article', '')): c
    for c in chunks if c.get('chunk_type') == 'article'
}
context_parts, sources, seen = [], [], set()
for hit in final_results:
    key = (hit.get('law_name', ''), hit.get('effective_date', ''), hit.get('article', ''))
    parent = article_parents.get(key)
    document = parent if parent is not None else hit
    if document['chunk_id'] in seen:
        continue
    seen.add(document['chunk_id'])
    sid = f'S{len(sources)+1}'
    context_parts.append(f"[{sid}] {document.get('article', '')} {document.get('title', '')}\n{document['content']}")
    sources.append({'source_id': sid, 'chunk_id': document['chunk_id'],
                    'article': document.get('article', ''), 'content': document['content']})
context = '\n\n'.join(context_parts)
print('근거 조문:', [s['article'] for s in sources])


In [ ]:
# 답변 생성은 별도 API 호출입니다. 필요할 때만 실행하세요.
llm = get_llm_model(max_tokens=1500)
response = await llm.ainvoke([
    ('system', '제공된 법령 근거만 사용해 한국어로 답하세요. 주장마다 [S1] 등의 출처를 표시하세요. '
               '근거가 부족하면 부족하다고 답하고 추측하지 마세요. 정의와 의무를 구분하고 예외 및 조건을 유지하세요.'),
    ('human', f'질문: {query}\n\n근거:\n{context}')
])
print(response.content)
